In [1]:
import os
import json
import time
from dotenv import load_dotenv
from google import genai
from google.genai import types
import random

In [2]:
load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY")

if not API_KEY:
    raise ValueError("Missing API Key! Check your .env file.")

In [3]:
client = genai.Client(api_key=API_KEY)

In [4]:
BATCHES = 1000
OUTPUT_FILE = "talangin_synthetic_templates.json"

In [ ]:
EXAMPLE_POOL = [
    {
        "format": "PARAGRAPH",
        "raw_text": "Sumpah ya [PERSON_1] lama bgt tf nya. Tagihan [ITEM_1] bulan ini udah keluar total [PRICE_1]. Karena kita berlima, patungan aja, masing-masing kena [PRICE_2]. Si [PERSON_2] kan kemaren udah nalangin [ITEM_2] gw [PRICE_3], jadi dia gausah tf.",
        "entities": [{"entity": "PERSON", "value": "[PERSON_1]"}, {"entity": "ITEM", "value": "[ITEM_1]"}, {"entity": "PRICE", "value": "[PRICE_1]"}, {"entity": "MULTIPLIER", "value": "berlima"}, {"entity": "MULTIPLIER", "value": "patungan"}, {"entity": "MULTIPLIER", "value": "masing-masing"}, {"entity": "PRICE", "value": "[PRICE_2]"}, {"entity": "PERSON", "value": "[PERSON_2]"}, {"entity": "ITEM", "value": "[ITEM_2]"}, {"entity": "PRICE", "value": "[PRICE_3]"}, {"entity": "PERSON", "value": "dia"}]
    },
    {
        "format": "LIST_ONLY",
        "raw_text": "Rekap patungan [ITEM_1] semalem:\n1. [PERSON_1]: [ITEM_2] [PRICE_1]\n2. [PERSON_2]: [PRICE_2]\n3. Gw: [PRICE_3]\nTotal [PRICE_4], bagi 3 jadi [PRICE_5]. Yg kurang bayar ke gw.",
        "entities": [{"entity": "MULTIPLIER", "value": "patungan"}, {"entity": "ITEM", "value": "[ITEM_1]"}, {"entity": "PERSON", "value": "[PERSON_1]"}, {"entity": "ITEM", "value": "[ITEM_2]"}, {"entity": "PRICE", "value": "[PRICE_1]"}, {"entity": "PERSON", "value": "[PERSON_2]"}, {"entity": "PRICE", "value": "[PRICE_2]"}, {"entity": "PRICE", "value": "[PRICE_3]"}, {"entity": "PRICE", "value": "[PRICE_4]"}, {"entity": "MULTIPLIER", "value": "bagi 3"}, {"entity": "PRICE", "value": "[PRICE_5]"}]
    },
    {
        "format": "SHORT_CHAT",
        "raw_text": "[PERSON_1] belum bayar [ITEM_1] kemaren, [PRICE_1] ya.",
        "entities": [{"entity":"PERSON","value":"[PERSON_1]"},{"entity":"ITEM","value":"[ITEM_1]"},{"entity":"PRICE","value":"[PRICE_1]"}]
    },
    {
        "format": "INCOMPLETE",
        "raw_text": "Inget ga kemaren [PERSON_1] pesen [ITEM_1]? Berapa tuh harganya lupa gw.",
        "entities": [{"entity":"PERSON","value":"[PERSON_1]"},{"entity":"ITEM","value":"[ITEM_1]"}]
    },
    {
        "format": "MIXED",
        "raw_text": "Eh guys, ini rincian [ITEM_1] kemaren ya:\n- [PERSON_1]: [PRICE_1]\n- [PERSON_2]: [PRICE_2]\nTolong tf ke gw ya berdua.",
        "entities": [{"entity":"ITEM","value":"[ITEM_1]"}, {"entity":"PERSON","value":"[PERSON_1]"}, {"entity":"PRICE","value":"[PRICE_1]"}, {"entity":"PERSON","value":"[PERSON_2]"}, {"entity":"PRICE","value":"[PRICE_2]"}, {"entity":"MULTIPLIER", "value":"berdua"}]
    },
    {
        "format": "CHAOTIC",
        "raw_text": "eh buset dah [PERSON_1] lu kmrn kan pesen [ITEM_1] gw lupa jdnya brp ya [PRICE_1] kl gasalah eh bukan deng wkwk au ah yg pntg tf aja dlu",
        "entities": [{"entity":"PERSON","value":"[PERSON_1]"}, {"entity":"ITEM","value":"[ITEM_1]"}, {"entity":"PRICE","value":"[PRICE_1]"}]
    }
]

In [ ]:
FORMATS = [
    ("LIST_ONLY: A strict, dry list or simulated receipt/nota format. No conversational filler.", 0.25),
    ("MIXED: Starts with a short conversational intro, followed by a bulleted or dashed list of items and prices.", 0.25),
    ("SHORT_CHAT: Extremely brief, 1 or 2 short sentences max. Straight to the point.", 0.15),
    ("INCOMPLETE: A natural chat, but intentionally missing some crucial information (e.g., mentioning the item but forgetting to put the exact [PRICE_X]).", 0.15),
    ("CHAOTIC: Stream-of-consciousness, mid-sentence topic jumps, typos intentional. Entity info buried and out of order.", 0.10),
    ("PARAGRAPH: A natural, long, flowing text message.", 0.10)
]

In [ ]:
SEEDS = {
    "personas": [
        "A broke college student ('Anak kos akhir bulan') who is desperate for the money.",
        "A passive-aggressive coworker who uses formal but slightly sarcastic Indonesian.",
        "A hyperactive Gen Z gamer using excessive gaming slang.",
        "A confused friend who is terrible at math and keeps miscalculating.",
        "A chill, wealthy friend ('Anak Jaksel') mixing English and Indonesian naturally.",
        "An angry roommate who is fed up with people not paying their debts.",
        "The 'Mom' of the group who organizes everything but is exhausted by her friends' laziness.",
        "An overly polite junior/underclassman who feels extremely awkward asking seniors for money.",
        "A dramatic friend who acts like they will literally starve to death if they don't get paid back today.",
        "A forgetful person who is suddenly remembering to collect a debt from 3 months ago.",
        "A gym bro/fitgirl who relates the bill to protein, macros, or their workout routine.",
        "A K-Pop stan who uses fandom slang (bias, comeback, photocard) while asking for the money.",
        "A fast-talker who uses maximum chat abbreviations (g, lu, y, krn, smpt, bgt, tp).",
        "A conspiracy theorist who complains that the app's admin fees and taxes are a scam.",
        "A romantic/galau friend who relates the group debt to their recent painful breakup.",
        "A foodie who gives a mini-review of the food while demanding the payment.",
        "A hustler/entrepreneur who tries to treat the split bill like a formal B2B business transaction.",
        "A sleepy friend who just woke up, is texting blindly, and makes multiple typos.",
        "A sarcastic troll who makes fun of how poor everyone else in the group is.",
        "A zodiac/astrology believer who blames the chaotic bill splitting on Mercury retrograde.",
        "A hyper-organized project manager who treats a casual hangout like a formal Scrum meeting."
    ],
    "scenarios": [
        "Splitting a late-night GoFood/GrabFood order (Martabak/Pecel Lele).",
        "Collecting money for a shared Netflix/Spotify/Canva premium account.",
        "Patungan for booking a Futsal or Badminton court.",
        "Calculating the split for a road trip (Bensin, Tol, and Snacks).",
        "Buying a shared birthday or wedding gift (Kado) for a mutual friend.",
        "Splitting an Airbnb or Hotel booking for an upcoming trip.",
        "A complicated Starbucks/Cafe order where everyone got different things.",
        "Splitting an overwhelming Shopee/Tokopedia haul to save on shipping (ongkir).",
        "Patungan for a friend's hospital bill or getting-well-soon fruit basket (Jenguk temen).",
        "Paying for shared internet/Wi-Fi at the boarding house (Indihome Kosan).",
        "Splitting a massive karaoke bill (Happy Puppy/Inul Vizta) with food and extra hours.",
        "Dividing the cost of a rental car (Sewa Mobil) for a weekend getaway.",
        "Patungan for concert tickets (e.g., Coldplay, Blackpink) bought under one account.",
        "Splitting a heavy drinking/bar tab where some people didn't drink alcohol.",
        "Buying group groceries for a BBQ/Grill night at someone's house.",
        "Splitting the cost of an escape room or amusement park ticket (Dufan).",
        "Patungan for a custom group jacket or t-shirt (Baju angkatan/PDH).",
        "Splitting a GoCar/GrabCar ride after a night out with multiple drop-offs.",
        "Buying a specific in-game item, battle pass, or game on Steam together.",
        "Splitting the bill at an All-You-Can-Eat (AYCE) place where some got the premium tier.",
        "Patungan for cat food or vet bills for the boarding house stray cat.",
        "Dividing the cost of raw materials/electronics for a college Capstone project."
    ],
    "complications": [
        "One person ordered something much more expensive and has to pay extra.",
        "Someone hasn't paid a previous debt, so it needs to be deducted from this bill.",
        "There is a weird tax, PB1, or service charge that makes the math confusing.",
        "Someone already transferred a DP (Down Payment) and only owes the rest.",
        "The speaker is threatening to kick someone out of the group if they don't pay.",
        "Someone claims they only ate 'a little bit' and shouldn't pay full price.",
        "The promo code failed or expired, so the price is much higher than expected.",
        "The driver got lost and they had to pay extra parking/toll fees in cash.",
        "Someone accidentally transferred the money to the wrong e-wallet/bank.",
        "The speaker lost the receipt and is trying to guess the exact prices from memory.",
        "Someone paid with a mix of ShopeePay coins/promo points and cash, confusing the math.",
        "One person is refusing to pay the n-% tax or n-% service charge out of principle.",
        "Someone left early and threw a random 50k bill on the table, which doesn't cover their share.",
        "Two people shared one food portion, so their part needs to be halved.",
        "The banking app (BCA Mobile/Livin) is offline/error, causing panic about who actually paid.",
        "Someone bought an item for their girlfriend/boyfriend who isn't in the group chat.",
        "The app charged a dynamic 'surge pricing' fee because it was raining.",
        "The speaker is rounding up aggressively to make a profit, and someone notices.",
        "Someone promised to pay 'next month after payday' but the speaker needs it now.",
        "A random extra item (like a plastic bag or dipping sauce) was added and nobody claims it.",
        "One person keeps saying 'bayarin dulu' (pay for me first) but has a terrible history of ghosting.",
        "The split is unequal because some items were meant for the whole group (e.g., sharing a snack platter)."
    ],
    "entity_constraints": [
        "Use all entity types freely.",
        "Do NOT include any [ITEM_X]. Refer to items vaguely ('itu', 'kemaren punya lu').",
        "Do NOT include [MULTIPLIER]. Just state each person's amount directly.",
        "Only mention one PERSON. The others are implied ('kalian', 'semua').",
        "PRICE is intentionally missing — the sender forgot or is asking."
    ]
}

In [ ]:
def get_random_seed():
    formats, weights = zip(*FORMATS)
    chosen_format = random.choices(formats, weights=weights, k=1)[0]
    
    return {
        "persona": random.choice(SEEDS["personas"]),
        "scenario": random.choice(SEEDS["scenarios"]),
        "complication": random.choice(SEEDS["complications"]),
        "format_style": chosen_format,
        "entity_constraint": random.choice(SEEDS["entity_constraints"])
    }

In [ ]:
SYSTEM_PROMPT = """You are an expert NLP data synthesizer and a native Indonesian speaker who is highly familiar with Gen Z and Millennial WhatsApp chat culture. 
Your task is to generate diverse Indonesian text messages regarding splitting group bills. 
Crucially, you must adapt your writing style to strictly match the requested format—seamlessly switching between highly complex, emotional chat paragraphs and extremely dry, rigid ledger lists as instructed.
Output ONLY a raw JSON array. Do not use markdown blocks."""

In [ ]:
def build_prompt(seed, anti_repeat_block, examples_block):
    return f"""Generate 3 unique JSON objects. 

### YOUR ASSIGNED CONTEXT FOR THIS BATCH:
- **Speaker Persona:** {seed['persona']}
- **Scenario:** {seed['scenario']}
- **Social Complication:** {seed['complication']}
- **OUTPUT FORMAT/STYLE:** {seed['format_style']}
- **Entity Constraint:** {seed['entity_constraint']}

{anti_repeat_block}

### CRITICAL INSTRUCTION: USE EXACT PLACEHOLDERS
Do NOT invent names, foods, or prices. You MUST use these exact bracketed placeholders in the text:
- People: [PERSON_1], [PERSON_2], [PERSON_3]
- Objects/Services: [ITEM_1], [ITEM_2]
- Money/Prices: [PRICE_1], [PRICE_2]
Note: Keep multipliers as natural words (e.g., 'bagi 3', 'patungan').

### CONSTRAINTS FOR REALISM & DIVERSITY:
1. ADAPT TO THE FORMAT STYLE (CRITICAL): The `format_style` overrides the persona. If the style is 'LIST_ONLY', strictly output a receipt-like list. ZERO conversational filler. No yapping, no storytelling.
2. You MUST adopt the Assigned Persona and Context perfectly, UNLESS restricted by the LIST_ONLY format.
3. TEMPLATE BUSTING (MANDATORY): NEVER start sentences with "Bro", "Eh", "Guys", "Yo", or "Wkwk". Jump straight into the thought.
4. ENTITIES TO EXTRACT: 
   - `PERSON`: The exact string '[PERSON_X]' or pronouns (dia, cowoknya).
   - `ITEM`: The exact string '[ITEM_X]'.
   - `PRICE`: The exact string '[PRICE_X]'.
   - `MULTIPLIER`: Phrases indicating division ('bagi 3', 'berempat', 'masing-masing').
   
### JSON SCHEMA ENFORCEMENT (CRITICAL)
Every single object in the JSON array MUST have exactly two keys:
1. "raw_text": The generated text message.
2. "entities": An array of objects mapping the placeholders, with keys "entity" and "value".
NEVER use "message" as a key. Do NOT forget the entities array.

### TARGET FORMAT EXAMPLE (USE THIS STRUCTURE):
[
{examples_block}
]

Start directly with '['."""

In [ ]:
def clean_json_response(text):
    """Strips markdown code blocks if the model ignores the system prompt."""
    text = text.strip()
    if text.startswith("```json"):
        text = text[7:]
    if text.startswith("```"):
        text = text[3:]
    if text.endswith("```"):
        text = text[:-3]
    return text.strip()

In [ ]:
master_dataset = []

if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
        master_dataset = json.load(f)
    print(f"Resuming from {len(master_dataset)} existing records.")

# MOD 6: Added top_k=40 to prevent degenerate loops
generation_config = types.GenerateContentConfig(
    system_instruction=SYSTEM_PROMPT,
    temperature=1.2, 
    top_p=0.95,
    top_k=40,
    max_output_tokens=8192,
)

for i in range(BATCHES):
    print(f"\n--- Generating batch {i + 1}/{BATCHES} ---")
    
    current_seed = get_random_seed()
    
    # MOD 5: Anti-repeat block extraction
    recent_openings = [r['raw_text'][:40] for r in master_dataset[-10:] if 'raw_text' in r]
    anti_repeat_block = ""
    if recent_openings:
        banned_list = "\n".join(f'- "{o}..."' for o in recent_openings)
        anti_repeat_block = f"### BANNED OPENING PATTERNS (do NOT start with these or similar):\n{banned_list}"
    
    # MOD 1: Fetch correct example dynamically
    format_key = current_seed['format_style'].split(":")[0]
    relevant_examples = [e for e in EXAMPLE_POOL if e.get('format') == format_key]
    
    if relevant_examples:
        chosen_example = random.choice(relevant_examples)
        # Strip the internal 'format' key before showing to the model
        display_example = {k: v for k, v in chosen_example.items() if k != 'format'}
        examples_block = json.dumps(display_example, indent=2)
    else:
        # Fallback if somehow missing
        examples_block = '  // Generate exactly as instructed by the schema without anchoring'

    print(f"Seeding -> Format: {format_key} | Constraint: {current_seed['entity_constraint'][:30]}...")
    
    # Build prompt dynamically
    seeded_prompt = build_prompt(current_seed, anti_repeat_block, examples_block)
    
    try:
        response_stream = client.models.generate_content_stream(
            model="gemma-4-31b-it", 
            contents=seeded_prompt,
            config=generation_config
        )
        
        raw_output = ""
        print("Generating JSON: ", end="", flush=True)

        for chunk in response_stream:
            if chunk.text:
                print(chunk.text, end="", flush=True) 
                raw_output += chunk.text              
        
        print("\n✅ Stream complete. Parsing JSON...")
        
        cleaned_output = clean_json_response(raw_output)
        batch_data = json.loads(cleaned_output)
        master_dataset.extend(batch_data)
        
        with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
            json.dump(master_dataset, f, indent=2, ensure_ascii=False)
            
        print(f"Success! Saved {len(batch_data)} records. Total: {len(master_dataset)}")
        
    except json.JSONDecodeError:
        print("❌ JSON Parsing Error. Skipping batch.")
    except Exception as e:
        print(f"⚠️ API Error: {e}")
        time.sleep(5) 
        
    time.sleep(1) 

print("\n🎉 Data generation complete!")


--- Generating batch 1/1000 ---
🌱 Seeding -> Format: LIST_ONLY: A strict,... | Persona: A gym bro/fitgirl wh... | Scenario: Buying a specific in...
Generating JSON: [
  {
    "raw_text": "Rincian Patungan:\n- [ITEM_1]: [PRICE_1]\n- [ITEM_2]: [PRICE_2]\nTotal bagi 3 ([PERSON_1], [PERSON_2], [PERSON_3]): [PRICE_1]",
    "entities": [
      {
        "entity": "MULTIPLIER",
        "value": "Patungan"
      },
      {
        "entity": "ITEM",
        "value": "[ITEM_1]"
      },
      {
        "entity": "PRICE",
        "value": "[PRICE_1]"
      },
      {
        "entity": "ITEM",
        "value": "[ITEM_2]"
      },
      {
        "entity": "PRICE",
        "value": "[PRICE_2]"
      },
      {
        "entity": "MULTIPLIER",
        "value": "bagi 3"
      },
      {
        "entity": "PERSON",
        "value": "[PERSON_1]"
      },
      {
        "entity": "PERSON",
        "value": "[PERSON_2]"
      },
      {
        "entity": "PERSON",
        "value": "[PERSON_3]"
      },


KeyboardInterrupt: 